# Lab 02 - 2 #

## Image classification on the Fashion-MNIST dataset using a ResNet-18 ##

**1**

* Create a custom `FMnistResNet18` class in which:
    * Download the pre-trained ResNet-18
    * Change the first and last layers

Specifically, the input and output layers of a pre-trained ResNet-18 need to be changed, since ResNet was originally designed for ImageNet competition, which was a color (3-channel) image classification task with 1000 classes. Fashon-MNIST, on the other hand, only contains 10 classes, and it’s images are in the grayscale (*i.e.*,1-channel).

In [1]:
import torch
import torch.optim
import torchvision

import torch.nn as nn
import torchvision.models as models

# from tqdm import tqdm
from tqdm.notebook import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights


# Hyperparameters.
LR = 3e-4
EPOCH = 3
BATCH_SIZE = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Currently using device:", DEVICE)

Currently using device: cuda


In [2]:
class FMnistResNet18(nn.Module):
    def __init__(self, in_channels=1):
        super(FMnistResNet18, self).__init__()

        # Load the pre-trained ResNet-18 model from torchvision.models.
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Change the input layer to take grayscale images, instead of RGB images.
        self.resnet.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Change the output layer to output 10 classes instead of 1000 classes.
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 10)

    def forward(self, x):
        return self.resnet(x)

# Test the network, and verify the layers.
test_my_resnet = FMnistResNet18()
# [N, C, H, W]: batch N, channels C, depth D, height H, width W.
dummy_input = torch.randn((32, 1, 244, 244))
output = test_my_resnet(dummy_input)

print(output.shape)

# print(test_my_resnet)

torch.Size([32, 10])


**2**

* Create DataLoaders
    * Hint: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
* Define the model
* Define the loss function
* Define the optimizer

In [3]:
# Dataset.
fashion_mnist = torchvision.datasets.FashionMNIST(download=True, train=True, root=".").train_data.float()

# Transformations.
data_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize((fashion_mnist.mean()/255,), (fashion_mnist.std()/255,))])

# Dataset
train_dataset = torchvision.datasets.FashionMNIST(download=True, train=True, root=".", transform=data_transform)
test_dataset = torchvision.datasets.FashionMNIST(download=True, train=False, root=".", transform=data_transform)

# DataLoaders.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Define the model.
model = FMnistResNet18().to(DEVICE)

# Define the loss function.
criterion = nn.CrossEntropyLoss()

# Define the optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

c:\Users\marco\miniconda3\Lib\site-packages\torchvision\datasets\mnist.py:76: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")


**3**

* Write the training step
    * Hint: https://pytorch.org/tutorials/beginner/introyt/trainingyt.html

In [4]:
losses = []

# Training step.
print(f"Start training on {DEVICE}")

for e in range(EPOCH):
    print('EPOCH {}:'.format(e + 1))
    e_loss = 0.0

    for i, data in (tepoch := tqdm(enumerate(train_loader), unit="batch", total=len(train_loader))):
        # Map data
        inputs, labels = data 
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass.
        outputs = model(inputs)

        # Compute the loss.
        loss = criterion(outputs, labels)

        # Backward pass.
        loss.backward()

        # Update the parameters.
        optimizer.step()

        e_loss += loss.item()
        tepoch.set_postfix(loss=loss.item())
        
    avg_epoch_loss = e_loss / len(train_loader)
    losses.append(avg_epoch_loss)
    print(f"Average Loss for EPOCH {e + 1}: {avg_epoch_loss:.4f}")

Start training on cuda
EPOCH 1:


  0%|          | 0/1200 [00:00<?, ?batch/s]

Average Loss for EPOCH 1: 0.2748
EPOCH 2:


  0%|          | 0/1200 [00:00<?, ?batch/s]

Average Loss for EPOCH 2: 0.1783
EPOCH 3:


  0%|          | 0/1200 [00:00<?, ?batch/s]

Average Loss for EPOCH 3: 0.1464


**4**

* Write the evaluation step

In [5]:
# Evaluation step.
t_loss = 0
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for i, data in (tepoch := tqdm(enumerate(test_loader), unit="batch", total=len(test_loader))):
        
        inputs, labels = data
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass
        outputs = model(inputs)
        
        # Calculate loss
        loss = criterion(outputs, labels)
        t_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = t_loss / len(test_loader)
accuracy = 100 * correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.2f}%")

  0%|          | 0/200 [00:00<?, ?batch/s]

Test Loss: 0.1981
Test Accuracy: 92.81%
